# Lab Day 10: Finetune Qwen3-VL-4B (LoRA) để định vị hư hại mặt đường

> Tinh chỉnh mô hình Vision-Language Model Qwen3-VL-4B bằng QLoRA trên tập dữ liệu ảnh dashcam.
> Môi trường thực thi khuyến nghị: **Google Colab T4 GPU** (16 GB VRAM).
> Xem chi tiết tài liệu tại `README.md`, `GUIDE.md`, `RUBRIC.md` và hoàn thiện báo cáo tại `REPORT.md`.

## 0. Môi trường (Colab T4)

In [ ]:
# 1. Kiểm tra cấu hình GPU trên Colab (yêu cầu GPU T4 16GB)
!nvidia-smi -L

# 2. Cài đặt các thư viện cần thiết theo yêu cầu của lab
!pip -q install -r requirements-colab.txt

# 3. Chạy kiểm thử tự động của bài lab để kiểm tra môi trường ban đầu
!python -m pytest -q

In [ ]:
# Cấu hình thông tin xác thực Kaggle qua Colab Secrets
# Hướng dẫn:
# 1. Mở menu hình chiếc chìa khoá bên trái giao diện Colab (Secrets)
# 2. Thêm Secret có tên 'KAGGLE_USERNAME' với giá trị là username Kaggle của bạn
# 3. Thêm Secret có tên 'KAGGLE_KEY' với giá trị là Kaggle API Token của bạn
import os
try:
    from google.colab import userdata
    os.environ['KAGGLE_USERNAME'] = userdata.get('KAGGLE_USERNAME', '')
    os.environ['KAGGLE_KEY'] = userdata.get('KAGGLE_KEY', '')
    print('✓ Đã nạp Kaggle credentials từ Colab Secrets thành công!')
except Exception as e:
    print('Lưu ý: Không tìm thấy Colab Secrets. Nếu chạy local hoặc đã có ~/.kaggle/kaggle.json thì bỏ qua.')

## 1. Dữ liệu: tải và chia split

In [ ]:
# Tải tập dữ liệu từ Kaggle và chia thành 3 tập train, val, test theo phiên quay (session)
# Phân chia theo phiên quay giúp ngăn chặn triệt để rò rỉ dữ liệu giữa train và test
!python scripts/prepare_data.py --kaggle

In [ ]:
# Đọc thông tin và trực quan hoá các mẫu dữ liệu
import json
from pathlib import Path
from PIL import Image, ImageDraw
import matplotlib.pyplot as plt

# Đọc thông tin phân bố split từ data/splits/meta.json
meta_path = Path('data/splits/meta.json')
if meta_path.exists():
    meta = json.loads(meta_path.read_text(encoding='utf-8'))
    print(f"Dataset version: {meta.get('dataset_version')}")
    print('Phân bố số lượng ảnh và nhãn theo từng split:')
    for split_name, cnt in meta.get('counts', {}).items():
        print(f" - {split_name:5s}: {cnt['images']:4d} ảnh | {cnt['boxes']}")

def read_jsonl(path):
    with open(path, encoding='utf-8') as f:
        return [json.loads(line) for line in f if line.strip()]

train_records = read_jsonl('data/splits/train.jsonl')
print(f"\nTổng số mẫu ảnh trong tập train: {len(train_records)}")

# Trực quan hoá 4 ảnh mẫu kèm bounding box (chuyển đổi từ YOLO [0, 1] sang toạ độ pixel)
fig, axes = plt.subplots(2, 2, figsize=(14, 8))
axes = axes.flatten()

color_map = {
    'pothole': 'red',     # ổ gà
    'crack': 'yellow',    # vết nứt
    'manhole': 'cyan'     # nắp cống
}

sample_records = [r for r in train_records if len(r['objects']) > 0][:4]
for idx, rec in enumerate(sample_records):
    img_path = Path('data') / rec['image_path']
    img = Image.open(img_path).convert('RGB')
    draw = ImageDraw.Draw(img)
    w_img, h_img = img.size
    
    for obj in rec['objects']:
        label = obj['label']
        xc, yc, bw, bh = obj['yolo']
        # YOLO (tâm xc, yc, kích thước bw, bh) -> Toạ độ pixel góc trên-trái và dưới-phải
        x1 = (xc - bw / 2) * w_img
        y1 = (yc - bh / 2) * h_img
        x2 = (xc + bw / 2) * w_img
        y2 = (yc + bh / 2) * h_img
        
        c = color_map.get(label, 'green')
        draw.rectangle([x1, y1, x2, y2], outline=c, width=3)
        draw.text((x1 + 2, max(0, y1 - 12)), label, fill=c)
        
    axes[idx].imshow(img)
    axes[idx].set_title(f"{rec['image_id']} ({len(rec['objects'])} objects)")
    axes[idx].axis('off')

plt.tight_layout()
plt.show()

## 2. Nhãn: YOLO → JSON thang 0–1000

In [ ]:
# Hàm chuyển đổi nhãn YOLO sang định dạng JSON grounding thang 0-1000

def yolo_to_box(label: str, yolo: list):
    """
    Chuyển nhãn YOLO [xc, yc, w, h] chuẩn hoá [0, 1] sang toạ độ nguyên [x1, y1, x2, y2]
    trên thang chuẩn 0-1000 (quy ước grounding của Qwen3-VL).
    """
    xc, yc, w, h = yolo
    x1 = (xc - w / 2) * 1000.0
    y1 = (yc - h / 2) * 1000.0
    x2 = (xc + w / 2) * 1000.0
    y2 = (yc + h / 2) * 1000.0
    
    # Làm tròn thành số nguyên và kẹp (clamp) trong khoảng [0, 1000]
    ix1 = max(0, min(1000, int(round(x1))))
    iy1 = max(0, min(1000, int(round(y1))))
    ix2 = max(0, min(1000, int(round(x2))))
    iy2 = max(0, min(1000, int(round(y2))))
    
    return {'label': label, 'bbox_2d': [ix1, iy1, ix2, iy2]}

def objects_to_answer(objects: list) -> str:
    """
    Chuyển danh sách objects thành chuỗi JSON:
    - Sắp xếp các box theo thứ tự (y1, x1): từ trên xuống dưới, từ trái sang phải
      để tạo thứ tự ổn định giúp mô hình học nhất quán.
    - Trả về JSON string không dấu cách thừa (separators=(',', ':')).
    """
    boxes = [yolo_to_box(o['label'], o['yolo']) for o in objects]
    boxes.sort(key=lambda b: (b['bbox_2d'][1], b['bbox_2d'][0]))
    return json.dumps(boxes, separators=(',', ':'))

# Định nghĩa câu lệnh Prompt chuẩn hướng dẫn mô hình phát hiện 3 lớp hư hại mặt đường
PROMPT = (
    'Detect road damages in this image. '
    'Identify potholes, cracks, and manholes. '
    'Return a JSON list of objects, each with \'label\' and \'bbox_2d\': [x1, y1, x2, y2] '
    'using normalized coordinates on a 0-1000 scale.'
)

# Kiểm tra thử hàm chuyển đổi với mẫu đầu tiên
sample = train_records[0]
sample_answer = objects_to_answer(sample['objects'])
print('Prompt:\n', PROMPT)
print('\nJSON Answer tương ứng của mẫu đầu tiên:\n', sample_answer)

## 3. E0: zero-shot

In [ ]:
# Nạp model gốc Qwen3-VL-4B lượng tử hoá 4-bit (NF4) và suy luận Zero-Shot trên tập test
import torch
from transformers import AutoProcessor, AutoModelForVision2Seq, BitsAndBytesConfig
from tqdm.auto import tqdm

# Cấu hình lượng tử hoá 4-bit NF4:
# LƯU Ý QUAN TRỌNG: llm_int8_skip_modules=['visual', 'lm_head'] giữ ViT và Projector ở fp16.
# Điều này giúp ta có thể huấn luyện đầy đủ Projector ở E2/E3 mà không gặp lỗi.
quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
    llm_int8_skip_modules=['visual', 'lm_head']
)

model_id = 'Qwen/Qwen3-VL-4B-Instruct'
print(f'Đang tải processor và model: {model_id} ...')

processor = AutoProcessor.from_pretrained(model_id)
# Đặt padding_side = 'left' cho Decoder khi suy luận batch
processor.tokenizer.padding_side = 'left'

model_e0 = AutoModelForVision2Seq.from_pretrained(
    model_id,
    quantization_config=quantization_config,
    torch_dtype=torch.float16,
    device_map={'': 0}
)
model_e0.eval()
print('✓ Đã tải xong model E0 lên GPU!')

In [ ]:
# Chạy suy luận Zero-Shot trên tập test (311 ảnh)
test_records = read_jsonl('data/splits/test.jsonl')
output_e0_path = Path('work/e0.jsonl')
output_e0_path.parent.mkdir(parents=True, exist_ok=True)

batch_size = 4
e0_predictions = []

print(f'Bắt đầu suy luận E0 trên {len(test_records)} ảnh test (batch size = {batch_size})...')

for i in tqdm(range(0, len(test_records), batch_size)):
    batch = test_records[i : i + batch_size]
    batch_images = [Image.open(Path('data') / r['image_path']).convert('RGB') for r in batch]
    
    batch_messages = [
        [
            {
                'role': 'user',
                'content': [
                    {'type': 'image', 'image': img},
                    {'type': 'text', 'text': PROMPT}
                ]
            }
        ]
        for img in batch_images
    ]
    
    inputs = processor.apply_chat_template(
        batch_messages,
        tokenize=True,
        add_generation_prompt=True,
        return_dict=True,
        return_tensors='pt',
        padding=True
    ).to('cuda')
    
    prompt_len = inputs.input_ids.shape[1]
    
    with torch.no_grad():
        out = model_e0.generate(
            **inputs,
            max_new_tokens=256,
            do_sample=False
        )
        
    # Cắt bỏ phần prompt, chỉ lấy các token mới sinh ra
    gen_tokens = out[:, prompt_len:]
    decoded = processor.batch_decode(gen_tokens, skip_special_tokens=True)
    
    for r, raw_text in zip(batch, decoded):
        e0_predictions.append({'image_id': r['image_id'], 'raw': raw_text.strip()})

# Ghi kết quả vào work/e0.jsonl
with open(output_e0_path, 'w', encoding='utf-8') as f:
    for row in e0_predictions:
        f.write(json.dumps(row, ensure_ascii=False) + '\n')

print(f'✓ Đã ghi {len(e0_predictions)} kết quả E0 vào {output_e0_path}')

# Xem thử 3 kết quả thô đầu tiên
print('\n--- Ví dụ 3 output thô từ E0 ---')
for p in e0_predictions[:3]:
    print(f"[{p['image_id']}]: {p['raw']}")

In [ ]:
# Chấm điểm E0 bằng trọng tài score.py
!python scripts/score.py --pred work/e0.jsonl --name E0 --out report/evidence/e0.json

In [ ]:
# Giải phóng bộ nhớ GPU sau khi chạy xong E0 để chuẩn bị huấn luyện
import gc
del model_e0
gc.collect()
torch.cuda.empty_cache()
print('✓ Đã giải phóng VRAM của E0!')

## 4. Dựng dữ liệu huấn luyện: chat template và mask loss

In [ ]:
# Hàm encode_example và collate_fn cho huấn luyện SFT với Mask Loss
from torch.utils.data import Dataset, DataLoader

class RoadDamageDataset(Dataset):
    def __init__(self, records, data_root='data'):
        self.records = records
        self.data_root = Path(data_root)

    def __len__(self):
        return len(self.records)

    def __getitem__(self, idx):
        rec = self.records[idx]
        image_path = self.data_root / rec['image_path']
        image = Image.open(image_path).convert('RGB')
        answer = objects_to_answer(rec['objects'])
        return image, PROMPT, answer

def encode_example(image, prompt, answer, processor):
    """
    Mã hoá một mẫu huấn luyện:
    - prompt_enc: chỉ phần prompt của user kèm generation prompt.
    - full_enc: toàn bộ chuỗi (user prompt + assistant answer).
    - labels: gán nhãn -100 cho toàn bộ token thuộc prompt và ảnh;
      chỉ tính loss trên các token của câu trả lời.
    """
    user_msg = {
        'role': 'user',
        'content': [
            {'type': 'image', 'image': image},
            {'type': 'text', 'text': prompt}
        ]
    }
    assistant_msg = {
        'role': 'assistant',
        'content': [{'type': 'text', 'text': answer}]
    }

    prompt_enc = processor.apply_chat_template(
        [user_msg],
        add_generation_prompt=True,
        tokenize=True,
        return_dict=True,
        return_tensors='pt'
    )
    
    full_enc = processor.apply_chat_template(
        [user_msg, assistant_msg],
        add_generation_prompt=False,
        tokenize=True,
        return_dict=True,
        return_tensors='pt'
    )

    n = prompt_enc.input_ids.shape[1]
    # Kiểm tra prompt là tiền tố chuẩn xác của chuỗi đầy đủ
    assert torch.equal(full_enc.input_ids[0, :n], prompt_enc.input_ids[0]), 'Prompt mismatch!'

    # Tạo labels để mask loss: chỉ tính loss trên câu trả lời của assistant
    labels = full_enc.input_ids.clone()
    labels[:, :n] = -100

    item = {
        'input_ids': full_enc.input_ids[0],
        'attention_mask': full_enc.attention_mask[0],
        'labels': labels[0]
    }
    if 'pixel_values' in full_enc:
        item['pixel_values'] = full_enc.pixel_values
    if 'image_grid_thw' in full_enc:
        item['image_grid_thw'] = full_enc.image_grid_thw
    if 'mm_token_type_ids' in full_enc:
        item['mm_token_type_ids'] = full_enc.mm_token_type_ids[0]

    return item

def collate_fn(batch_items, pad_token_id):
    """
    Hàm ghép lô (collate):
    - Đệm (pad) input_ids, attention_mask, labels đến độ dài lớn nhất trong batch.
    - Nối (cat) pixel_values và image_grid_thw theo chiều 0 (KHÔNG đệm).
    """
    max_len = max(len(x['input_ids']) for x in batch_items)
    
    input_ids_list = []
    attention_mask_list = []
    labels_list = []
    mm_token_types = []
    
    for x in batch_items:
        pad_len = max_len - len(x['input_ids'])
        input_ids_list.append(torch.cat([x['input_ids'], torch.full((pad_len,), pad_token_id, dtype=torch.long)]))
        attention_mask_list.append(torch.cat([x['attention_mask'], torch.zeros(pad_len, dtype=torch.long)]))
        labels_list.append(torch.cat([x['labels'], torch.full((pad_len,), -100, dtype=torch.long)]))
        if 'mm_token_type_ids' in x:
            mm_token_types.append(torch.cat([x['mm_token_type_ids'], torch.zeros(pad_len, dtype=torch.long)]))

    batch = {
        'input_ids': torch.stack(input_ids_list),
        'attention_mask': torch.stack(attention_mask_list),
        'labels': torch.stack(labels_list),
    }
    if mm_token_types:
        batch['mm_token_type_ids'] = torch.stack(mm_token_types)
        
    if 'pixel_values' in batch_items[0]:
        batch['pixel_values'] = torch.cat([x['pixel_values'] for x in batch_items], dim=0)
    if 'image_grid_thw' in batch_items[0]:
        batch['image_grid_thw'] = torch.cat([x['image_grid_thw'] for x in batch_items], dim=0)
        
    return batch

# Kiểm tra thử encode_example với mẫu đầu tiên
sample_img = Image.open(Path('data') / train_records[0]['image_path']).convert('RGB')
sample_enc = encode_example(sample_img, PROMPT, objects_to_answer(train_records[0]['objects']), processor)

# In số token ảnh và giải mã phần câu trả lời (token labels != -100)
active_labels = sample_enc['labels'][sample_enc['labels'] != -100]
print(f"Tổng chiều dài token chuỗi: {len(sample_enc['input_ids'])}")
print(f"Số token thực sự tính loss: {len(active_labels)}")
print('Nội dung giải mã phần tính loss:')
print(processor.tokenizer.decode(active_labels))

## 5. Setup LoRA / đóng băng cho E1, E2, E3

In [ ]:
# Cấu hình LoRA và thiết lập đóng băng/mở khoá các khối theo GUIDE bước 5
from peft import LoraConfig, get_peft_model

def apply_setup(model, setup: str, r: int = 16, alpha: int = 32, dropout: float = 0.05):
    """
    Áp dụng cấu hình đóng băng và gắn LoRA:
    - setup == 'E1': LoRA chỉ trên LLM (252 modules: q,k,v,o_proj và gate,up,down_proj). Freeze ViT & Projector.
    - setup == 'E2': E1 + mở toàn bộ Projector (visual.merger và visual.deepstack_merger_list, chuyển sang float32).
    - setup == 'E3': E2 + LoRA trên Attention & MLP của ViT (thêm 96 modules: qkv, proj, linear_fc1, linear_fc2).
    """
    setup = setup.upper()
    print(f'\n--- Thiết lập mô hình cho {setup} ---')
    
    # 1. Đóng băng toàn bộ tham số ban đầu
    for param in model.parameters():
        param.requires_grad = False
        
    # Kích hoạt require_grads cho input để dùng gradient checkpointing
    model.enable_input_require_grads()
    
    # 2. Xác định các target modules cho LoRA
    if setup in ('E1', 'E2'):
        # Chỉ gắn LoRA lên LLM (36 layers * 7 modules = 252 modules)
        target_regex = r'.*language_model.*\.(q_proj|k_proj|v_proj|o_proj|gate_proj|up_proj|down_proj)$'
    elif setup == 'E3':
        # Gắn LoRA lên cả LLM (252) và ViT (96 modules: 24 blocks * 4 modules) -> Tổng 348 modules
        target_regex = r'.*(language_model.*\.(q_proj|k_proj|v_proj|o_proj|gate_proj|up_proj|down_proj)|visual\.blocks.*\.(qkv|proj|linear_fc1|linear_fc2))$'
    else:
        raise ValueError(f'Setup không hợp lệ: {setup}')
        
    lora_config = LoraConfig(
        r=r,
        lora_alpha=alpha,
        lora_dropout=dropout,
        target_modules=target_regex,
        bias='none',
        task_type='CAUSAL_LM'
    )
    
    peft_model = get_peft_model(model, lora_config)
    
    # 3. Mở khoá Projector cho E2 và E3
    if setup in ('E2', 'E3'):
        projector_params_count = 0
        for name, param in peft_model.named_parameters():
            if 'visual.merger' in name or 'visual.deepstack_merger_list' in name:
                param.data = param.data.to(torch.float32)
                param.requires_grad = True
                projector_params_count += param.numel()
        print(f'✓ Đã mở toàn bộ Projector ({projector_params_count:,} tham số) ở kiểu float32!')
        
    # Đếm và in tổng số tham số có thể huấn luyện
    trainable_params = sum(p.numel() for p in peft_model.parameters() if p.requires_grad)
    all_params = sum(p.numel() for p in peft_model.parameters())
    print(f'Tổng tham số trainable: {trainable_params:,} / {all_params:,} ({100 * trainable_params / all_params:.2f}%)')
    
    return peft_model

## 6. Vòng huấn luyện

In [ ]:
# Vòng huấn luyện chuẩn: AdamW, Cosine LR scheduler, FP16 GradScaler, Gradient Checkpointing
import time
import math
from torch.amp import autocast, GradScaler
from transformers import get_cosine_schedule_with_warmup

def train_experiment(setup: str, lora_r: int = 16, lora_alpha: int = 32, max_steps: int = 300, lr: float = 2e-4):
    """
    Hàm huấn luyện hoàn chỉnh cho từng thí nghiệm E1, E2, E3:
    - Tải model base 4-bit sạch
    - Áp dụng cấu hình apply_setup
    - Huấn luyện 300 bước tối ưu (batch 2 x grad_accum 4)
    - Lưu adapter LoRA và lưu riêng projector (cho E2, E3)
    - Đo VRAM đỉnh và thời gian chạy
    """
    setup = setup.upper()
    print(f'\n=======================================================')
    print(f'   BẮT ĐẦU HUẤN LUYỆN THÍ NGHIỆM {setup}')
    print(f'=======================================================')
    
    # Dọn dẹp GPU trước khi bắt đầu
    gc.collect()
    torch.cuda.empty_cache()
    torch.cuda.reset_peak_memory_stats()
    start_time = time.time()
    
    # Nạp base model 4-bit
    base_model = AutoModelForVision2Seq.from_pretrained(
        model_id,
        quantization_config=quantization_config,
        torch_dtype=torch.float16,
        device_map={'': 0}
    )
    
    model = apply_setup(base_model, setup=setup, r=lora_r, alpha=lora_alpha)
    model.train()
    
    # Bật gradient checkpointing để tiết kiệm VRAM
    model.gradient_checkpointing_enable(gradient_checkpointing_kwargs={'use_reentrant': False})
    model.config.use_cache = False
    
    # Bộ tối ưu hoá AdamW cho tất cả tham số requires_grad
    trainable_params_list = [p for p in model.parameters() if p.requires_grad]
    trainable_params_count = sum(p.numel() for p in trainable_params_list)
    optimizer = torch.optim.AdamW(trainable_params_list, lr=lr, weight_decay=0.0)
    
    # Lịch LR: 5% warmup (15 bước), sau đó giảm theo Cosine về 0
    warmup_steps = int(0.05 * max_steps)
    scheduler = get_cosine_schedule_with_warmup(
        optimizer,
        num_warmup_steps=warmup_steps,
        num_training_steps=max_steps
    )
    
    scaler = GradScaler('cuda')
    
    # Tạo Dataset và DataLoader
    dataset = RoadDamageDataset(train_records)
    batch_size = 2
    grad_accum_steps = 4
    pad_id = processor.tokenizer.pad_token_id if processor.tokenizer.pad_token_id is not None else 0
    
    def custom_collate(batch):
        encoded = [encode_example(img, prompt, ans, processor) for img, prompt, ans in batch]
        return collate_fn(encoded, pad_token_id=pad_id)
        
    loader = DataLoader(
        dataset,
        batch_size=batch_size,
        shuffle=True,
        collate_fn=custom_collate
    )
    
    step = 0
    accum_step = 0
    running_loss = 0.0
    optimizer.zero_grad()
    
    save_dir = Path(f'work/{setup.lower()}')
    save_dir.mkdir(parents=True, exist_ok=True)
    
    pbar = tqdm(total=max_steps, desc=f'Training {setup}')
    data_iter = iter(loader)
    
    while step < max_steps:
        try:
            batch = next(data_iter)
        except StopIteration:
            data_iter = iter(loader)
            batch = next(data_iter)
            
        batch = {k: v.to('cuda') for k, v in batch.items()}
        
        with autocast('cuda', dtype=torch.float16):
            outputs = model(**batch)
            loss = outputs.loss / grad_accum_steps
            
        if not math.isfinite(loss.item()):
            raise ValueError(f'Loss không hữu hạn tại bước {step}: {loss.item()}. Dừng huấn luyện!')
            
        scaler.scale(loss).backward()
        accum_step += 1
        running_loss += loss.item() * grad_accum_steps
        
        if accum_step % grad_accum_steps == 0:
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(trainable_params_list, 1.0)
            scaler.step(optimizer)
            scaler.update()
            optimizer.zero_grad()
            scheduler.step()
            
            step += 1
            pbar.update(1)
            pbar.set_postfix({'loss': f'{running_loss / grad_accum_steps:.4f}', 'lr': f'{scheduler.get_last_lr()[0]:.2e}'})
            running_loss = 0.0
            
            # Mỗi 100 bước hoặc khi kết thúc: lưu adapter và projector
            if step % 100 == 0 or step == max_steps:
                model.save_pretrained(save_dir / 'adapter')
                if setup in ('E2', 'E3'):
                    # Lưu riêng các tham số của Projector đã được train
                    proj_state = {
                        name: param.data.cpu()
                        for name, param in model.named_parameters()
                        if 'visual.merger' in name or 'visual.deepstack_merger_list' in name
                    }
                    torch.save(proj_state, save_dir / 'projector.pt')
                    
    pbar.close()
    
    # Đo đạc tài nguyên
    peak_vram_gb = torch.cuda.max_memory_allocated() / (1024 ** 3)
    train_minutes = (time.time() - start_time) / 60.0
    
    run_meta = {
        'trainable_params': trainable_params_count,
        'peak_vram_gb': round(peak_vram_gb, 3),
        'train_minutes': round(train_minutes, 2)
    }
    with open(save_dir / 'run.json', 'w', encoding='utf-8') as f:
        json.dump(run_meta, f, indent=2)
        
    print(f'\n✓ Đã hoàn thành {setup}:')
    print(f' - Trainable params: {trainable_params_count:,}')
    print(f' - Peak VRAM: {peak_vram_gb:.2f} GB')
    print(f' - Thời gian huấn luyện: {train_minutes:.2f} phút')
    print(f' - Đã lưu checkpoint tại: {save_dir}')
    
    # Giải phóng bộ nhớ
    del model, base_model, optimizer, scheduler, scaler
    gc.collect()
    torch.cuda.empty_cache()
    
    return run_meta

## 7. Suy luận và chấm E1–E3

In [ ]:
# Hàm chạy suy luận và chấm điểm cho các thí nghiệm E1, E2, E3
from peft import PeftModel
import subprocess

def evaluate_experiment(setup: str, run_meta: dict = None, evidence_filename: str = None):
    """
    Nạp base model + nạp adapter LoRA + nạp projector (nếu E2/E3),
    chạy suy luận trên toàn bộ 311 ảnh của tập test, ghi work/{setup}.jsonl
    và chạy script chấm điểm scripts/score.py.
    """
    setup = setup.upper()
    save_dir = Path(f'work/{setup.lower()}')
    if run_meta is None:
        meta_file = save_dir / 'run.json'
        if meta_file.exists():
            run_meta = json.loads(meta_file.read_text(encoding='utf-8'))
        else:
            run_meta = {'trainable_params': 1, 'peak_vram_gb': 1.0, 'train_minutes': 1.0}

    print(f'\n--- Đang tải mô hình đã finetune cho {setup} để suy luận ---')
    gc.collect()
    torch.cuda.empty_cache()
    
    # 1. Nạp base model 4-bit
    base_model = AutoModelForVision2Seq.from_pretrained(
        model_id,
        quantization_config=quantization_config,
        torch_dtype=torch.float16,
        device_map={'': 0}
    )
    
    # 2. Nạp LoRA adapter
    model = PeftModel.from_pretrained(base_model, save_dir / 'adapter')
    
    # 3. Nạp lại Projector nếu là E2 hoặc E3
    if setup in ('E2', 'E3'):
        proj_path = save_dir / 'projector.pt'
        if proj_path.exists():
            proj_state = torch.load(proj_path, map_location='cuda')
            model.load_state_dict(proj_state, strict=False)
            print('✓ Đã nạp lại projector weights thành công!')
        else:
            print('Cảnh báo: Không tìm thấy file projector.pt!')
            
    model.eval()
    
    # 4. Chạy suy luận trên tập test
    test_records = read_jsonl('data/splits/test.jsonl')
    pred_path = Path(f'work/{setup.lower()}.jsonl')
    pred_path.parent.mkdir(parents=True, exist_ok=True)
    
    batch_size = 4
    preds = []
    print(f'Đang suy luận {setup} trên {len(test_records)} ảnh test...')
    
    for i in tqdm(range(0, len(test_records), batch_size)):
        batch = test_records[i : i + batch_size]
        batch_images = [Image.open(Path('data') / r['image_path']).convert('RGB') for r in batch]
        
        batch_messages = [
            [
                {
                    'role': 'user',
                    'content': [
                        {'type': 'image', 'image': img},
                        {'type': 'text', 'text': PROMPT}
                    ]
                }
            ]
            for img in batch_images
        ]
        
        inputs = processor.apply_chat_template(
            batch_messages,
            tokenize=True,
            add_generation_prompt=True,
            return_dict=True,
            return_tensors='pt',
            padding=True
        ).to('cuda')
        
        prompt_len = inputs.input_ids.shape[1]
        with torch.no_grad():
            out = model.generate(**inputs, max_new_tokens=256, do_sample=False)
            
        gen_tokens = out[:, prompt_len:]
        decoded = processor.batch_decode(gen_tokens, skip_special_tokens=True)
        
        for r, text in zip(batch, decoded):
            preds.append({'image_id': r['image_id'], 'raw': text.strip()})
            
    with open(pred_path, 'w', encoding='utf-8') as f:
        for row in preds:
            f.write(json.dumps(row, ensure_ascii=False) + '\n')
            
    # Giải phóng model
    del model, base_model
    gc.collect()
    torch.cuda.empty_cache()
    
    # 5. Chạy chấm điểm với scripts/score.py
    if evidence_filename is None:
        evidence_filename = f'{setup.lower()}.json'
    evidence_path = f'report/evidence/{evidence_filename}'
    
    cmd = [
        'python', 'scripts/score.py',
        '--pred', str(pred_path),
        '--name', setup,
        '--out', evidence_path,
        '--meta', f"trainable_params={run_meta['trainable_params']}",
        '--meta', f"peak_vram_gb={run_meta['peak_vram_gb']}",
        '--meta', f"train_minutes={run_meta['train_minutes']}"
    ]
    print(f'Đang chấm điểm bằng lệnh: {" ".join(cmd)}')
    subprocess.run(cmd, check=True)
    print(f'✓ Đã lưu kết quả đánh giá tại: {evidence_path}')

In [ ]:
# Thực hiện huấn luyện và đánh giá lần lượt E1, E2, E3
# Lưu ý: Mỗi setup chạy khoảng 20-30 phút trên Colab T4 (300 bước).
# Bạn có thể chạy từng ô hoặc lần lượt từng hàm:

# 1. Chạy E1 (LoRA LLM, Freeze ViT & Projector)
meta_e1 = train_experiment('E1')
evaluate_experiment('E1', meta_e1)

# 2. Chạy E2 (E1 + Train Full Projector)
meta_e2 = train_experiment('E2')
evaluate_experiment('E2', meta_e2)

# 3. Chạy E3 (E2 + LoRA trên ViT)
meta_e3 = train_experiment('E3')
evaluate_experiment('E3', meta_e3)

## 8. E4 (ablation tự chọn) và E5 (tuỳ chọn, GRPO)

In [ ]:
# ==========================================================
# THÍ NGHIỆM E4 (Ablation Study)
# Ở đây ta chọn biến thể: LoRA Rank r (So sánh r=64 với r=16 ở E1)
# Chỉ thay đổi ĐÚNG 1 biến: rank r=64 (lora_alpha=128), giữ nguyên mọi tham số khác
# ==========================================================

print('Chạy Ablation E4 với rank r = 64 (alpha = 128)...')
meta_e4 = train_experiment('E1', lora_r=64, lora_alpha=128, max_steps=300)

# Đánh giá và lưu bằng chứng theo đúng tên chuẩn e4_rank_64.json
evaluate_experiment('E1', meta_e4, evidence_filename='e4_rank_64.json')
print('✓ Đã tạo file bằng chứng E4 tại: report/evidence/e4_rank_64.json')

In [ ]:
# ==========================================================
# THÍ NGHIỆM E5 (+10 điểm Bonus): GRPO với reward IoU
# ==========================================================
from scripts.score import parse_output, iou_xyxy

def compute_iou_reward(completion: str, gt_boxes: list) -> float:
    """
    Hàm tính reward kiểm chứng được cho GRPO theo GUIDE bước 9:
    - Nếu JSON hỏng -> 0.0
    - Nếu không có gt và không có pred -> 1.0
    - Ghép đôi tham lam theo IoU giảm dần, chỉ nhận IoU >= 0.5
    - Phạt nếu sinh thừa box bịa đặt (qua mẫu số max(n_gt, n_pred, 1))
    """
    pred = parse_output(completion)
    if pred is None:
        return 0.0
    if len(gt_boxes) == 0 and len(pred) == 0:
        return 1.0
    if len(gt_boxes) == 0 or len(pred) == 0:
        return 0.2
        
    pairs = []
    for i, g in enumerate(gt_boxes):
        for j, p in enumerate(pred):
            if g['label'] == p['label']:
                iou_val = iou_xyxy(g['bbox'], p['bbox'])
                if iou_val >= 0.5:
                    pairs.append((iou_val, i, j))
                    
    pairs.sort(reverse=True, key=lambda x: x[0])
    used_g, used_p = set(), set()
    total_iou = 0.0
    for iou_val, i, j in pairs:
        if i not in used_g and j not in used_p:
            used_g.add(i)
            used_p.add(j)
            total_iou += iou_val
            
    reward_score = 0.2 + 0.8 * (total_iou / max(len(gt_boxes), len(pred), 1))
    return float(reward_score)

# Kiểm tra unit test cho hàm reward
perfect_gt = [{'label': 'pothole', 'bbox': (400, 400, 600, 600)}]
print('Test 1 - Dự đoán hoàn hảo (IoU = 1.0) :', compute_iou_reward('[{"label":"pothole","bbox_2d":[400,400,600,600]}]', perfect_gt))
print('Test 2 - JSON lỗi cú pháp             :', compute_iou_reward('invalid json text', perfect_gt))
print('Test 3 - Cả hai đều rỗng (ảnh sạch)   :', compute_iou_reward('[]', []))
print('Test 4 - Sai nhãn (crack thay pothole):', compute_iou_reward('[{"label":"crack","bbox_2d":[400,400,600,600]}]', perfect_gt))

In [ ]:
# Kiểm tra toàn bộ bài nộp trước khi commit và nộp bài
# Đảm bảo lệnh này chạy ra '0 FAIL'
!python scripts/check_submission.py